In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd

DATASET_PATH = "/content/drive/MyDrive/IITG dataset/cleaned_dataset"

DATA_PATH = os.path.join(DATASET_PATH, "data")
METADATA_PATH = os.path.join(DATASET_PATH, "metadata.csv")

metadata = pd.read_csv(METADATA_PATH)

print("Metadata shape:", metadata.shape)
print("Columns:")
print(metadata.columns.tolist())

Metadata shape: (7565, 10)
Columns:
['type', 'start_time', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity', 'Re', 'Rct']


In [3]:
print("Number of batteries:", metadata["battery_id"].nunique())
print("Number of tests:", metadata["test_id"].nunique())
print("Number of files:", metadata["filename"].nunique())

print("\nBattery types:")
print(metadata["type"].value_counts())

Number of batteries: 34
Number of tests: 616
Number of files: 7565

Battery types:
type
charge       2815
discharge    2794
impedance    1956
Name: count, dtype: int64


In [5]:
# DAY 9 - Target Construction: Capacity

target_df = metadata.copy()

# Convert Capacity to numeric
target_df["Capacity_numeric"] = pd.to_numeric(
    target_df["Capacity"],
    errors="coerce"
)

print("Total records:", len(target_df))
print("Valid Capacity values:", target_df["Capacity_numeric"].notna().sum())
print("Invalid/Non-numeric Capacity values:", target_df["Capacity_numeric"].isna().sum())

print("\nCapacity statistics:")
display(target_df["Capacity_numeric"].describe())

print("\nCapacity by battery type:")
display(
    target_df.groupby("type")["Capacity_numeric"]
    .agg(["count", "min", "max", "mean"])
)

Total records: 7565
Valid Capacity values: 2769
Invalid/Non-numeric Capacity values: 4796

Capacity statistics:


,Capacity_numeric
count,2769.000000
mean,1.326543
std,0.472517
min,0.000000
25%,1.150286
50%,1.428065
75%,1.673645
max,2.640149



Capacity by battery type:


,count,min,max,mean
type,,,,
charge,0,NaN,NaN,NaN
discharge,2769,0.0,2.640149,1.326543
impedance,0,NaN,NaN,NaN


In [6]:
# DAY 9 - Check temporal ordering information

print("start_time data type:")
print(metadata["start_time"].dtype)

print("\nFirst 10 start_time values:")
display(metadata[["battery_id", "test_id", "type", "start_time", "Capacity"]].head(10))

start_time data type:
object

First 10 start_time values:


,battery_id,test_id,type,start_time,Capacity
0,B0047,0,discharge,[2010. 7. 21. 15. 0. ...,1.6743047446975208
1,B0047,1,impedance,[2010. 7. 21. 16. 53. ...,NaN
2,B0047,2,charge,[2010. 7. 21. 17. 25. ...,NaN
3,B0047,3,impedance,[2010 7 21 20 31 5],NaN
4,B0047,4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,1.5243662105099023
5,B0047,5,charge,[2010. 7. 21. 22. 38. ...,NaN
6,B0047,6,discharge,[2.010e+03 7.000e+00 2.200e+01 1.000e+00 4.000...,1.5080762969973425
7,B0047,7,charge,[2010. 7. 22. 3. 14. ...,NaN
8,B0047,8,discharge,[2010. 7. 22. 6. 16. ...,1.4835577960067696
9,B0047,9,charge,[2010. 7. 22. 7. 50. ...,NaN


In [7]:
# DAY 9 - Verify chronological sequence using test_id

sequence_check = (
    metadata.groupby("battery_id")["test_id"]
    .agg(["min", "max", "count", "nunique"])
    .reset_index()
)

sequence_check["expected_count"] = (
    sequence_check["max"] - sequence_check["min"] + 1
)

sequence_check["has_gap"] = (
    sequence_check["count"] != sequence_check["expected_count"]
)

print("Battery sequence summary:")
display(sequence_check.head(10))

print("\nBatteries with possible test-ID gaps:",
      sequence_check["has_gap"].sum())

Battery sequence summary:


,battery_id,min,max,count,nunique,expected_count,has_gap
0,B0005,0,615,616,616,616,False
1,B0006,0,615,616,616,616,False
2,B0007,0,615,616,616,616,False
3,B0018,0,318,319,319,319,False
4,B0025,0,79,80,80,80,False
5,B0026,0,79,80,80,80,False
6,B0027,0,79,80,80,80,False
7,B0028,0,79,80,80,80,False
8,B0029,0,96,97,97,97,False
9,B0030,0,96,97,97,97,False



Batteries with possible test-ID gaps: 0


In [8]:
# DAY 9 - Temporal Train / Validation / Test Split

split_df = metadata.copy()

# Keep records ordered by battery and test sequence
split_df = split_df.sort_values(
    ["battery_id", "test_id"]
).reset_index(drop=True)

def assign_split(group):
    n = len(group)

    train_end = int(0.70 * n)
    val_end = int(0.85 * n)

    group = group.copy()

    group["split"] = "test"

    group.iloc[:train_end, group.columns.get_loc("split")] = "train"
    group.iloc[train_end:val_end, group.columns.get_loc("split")] = "validation"

    return group

split_df = (
    split_df
    .groupby("battery_id", group_keys=False)
    .apply(assign_split)
)

print("Split distribution:")
print(split_df["split"].value_counts())

print("\nSplit distribution by battery:")
display(
    pd.crosstab(
        split_df["battery_id"],
        split_df["split"]
    ).head(10)
)

/tmp/ipykernel_1822/1145274366.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_split)


Split distribution:
split
train         5281
test          1149
validation    1135
Name: count, dtype: int64

Split distribution by battery:


split,test,train,validation
battery_id,,,
B0005,93,431,92
B0006,93,431,92
B0007,93,431,92
B0018,48,223,48
B0025,12,56,12
B0026,12,56,12
B0027,12,56,12
B0028,12,56,12
B0029,15,67,15


In [9]:
# DAY 9 - Leakage-Safe Temporal Split Check

leakage_check = (
    split_df.groupby(["battery_id", "split"])["test_id"]
    .agg(["min", "max"])
    .reset_index()
)

display(leakage_check.head(15))

# Check chronological ordering
problems = []

for battery, group in split_df.groupby("battery_id"):
    train_max = group.loc[group["split"] == "train", "test_id"].max()
    val_min = group.loc[group["split"] == "validation", "test_id"].min()
    val_max = group.loc[group["split"] == "validation", "test_id"].max()
    test_min = group.loc[group["split"] == "test", "test_id"].min()

    if not (train_max < val_min and val_max < test_min):
        problems.append(battery)

print("\nBatteries with temporal leakage:", len(problems))

if len(problems) == 0:
    print("✓ No temporal ordering violations detected.")
else:
    print("Problem batteries:", problems)

,battery_id,split,min,max
0,B0005,test,523,615
1,B0005,train,0,430
2,B0005,validation,431,522
3,B0006,test,523,615
4,B0006,train,0,430
5,B0006,validation,431,522
6,B0007,test,523,615
7,B0007,train,0,430
8,B0007,validation,431,522
9,B0018,test,271,318



Batteries with temporal leakage: 0
✓ No temporal ordering violations detected.
